# XC And SCF Mixing

This notebook compares exchange-only LDA against LDA exchange-correlation, then compares linear mixing against Pulay DIIS. These are still toy Gaussian-pseudopotential systems, but the diagnostics are the same shape we need for more realistic DFT.

In [ ]:
import matplotlib.pyplot as plt

from mlx_atomistic.dft import DFTSystem, DiracExchange, LDAExchangeCorrelation, SCFConfig, run_scf

The system has two smooth attractive centers and two electrons. The grid is small enough that this notebook runs quickly, but large enough to show useful residual traces.

In [ ]:
system = DFTSystem(
    cell=[8.0, 8.0, 8.0],
    grid_shape=(6, 6, 6),
    electron_count=2.0,
    centers=[[3.4, 4.0, 4.0], [4.6, 4.0, 4.0]],
    amplitudes=[-2.0, -2.0],
    widths=[0.8, 0.8],
)

cases = []
for xc_name, xc in [("exchange-only", DiracExchange()), ("lda-xc", LDAExchangeCorrelation())]:
    for mixer in ["linear", "diis"]:
        config = SCFConfig(max_iterations=5, tolerance=1e-8, mixer=mixer, solver="dense", seed=17)
        result = run_scf(system, config=config, xc_functional=xc)
        cases.append({"xc": xc_name, "mixer": mixer, "result": result})

[{k: v for k, v in case.items() if k != "result"} | case["result"].to_dict() for case in cases]

The residual trace shows how much the density changes each iteration. DIIS may not always win on a tiny toy case, but this makes the mixer behavior visible and testable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

labels = []
energies = []
for case in cases:
    label = f"{case['xc']} / {case['mixer']}"
    labels.append(label)
    energies.append(case["result"].total_energy)
    history = case["result"].history
    axes[0].semilogy(
        [row["iteration"] for row in history],
        [row["density_residual"] for row in history],
        marker="o",
        label=label,
    )

axes[0].set_xlabel("iteration")
axes[0].set_ylabel("density residual")
axes[0].legend(fontsize=8)
axes[0].set_title("SCF residual")

axes[1].barh(labels, energies)
axes[1].set_xlabel("final total energy / hartree")
axes[1].set_title("XC and mixer comparison");